In [ ]:
import datetime as dt
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

from QuantStudio.Core.QSObject import Panel
from QuantStudio.Factor.HDF5DB import HDF5DB
from QuantStudio.Tools.Visualization import qs_help

# 策略回测

策略回测用于评估交易策略在历史数据上的表现。QuantStudio 的策略回测功能位于 `BackTest.Strategy` 子模块中，建立在回测基本框架之上。

相关概念见 [基本框架](基本框架.ipynb)，底层调度与扩展接口见 [原理和细节](原理和细节.ipynb)。

## 模块概览

`QuantStudio.BackTest.Strategy` 提供以下核心组件：

| 组件 | 类型 | 功能 |
|------|------|------|
| `MakeAccount` | `PanelOperator` | 账户创建算子，基于交易信号撮合成交、更新持仓和现金 |
| `MakeStrategy` | `PanelOperator`（可继承） | 策略创建算子，通过重写 `genSignal` 方法实现自定义策略逻辑 |
| `AccountStats` | `BTNode` | 账户统计节点，计算净值、收益率和绩效统计指标 |
| `AccountReport` | `ReportNode` | 账户报告节点，基于统计结果生成 HTML 报告 |
| `CalcPortfolioNV` | `PanelOperator` | 投资组合净值计算算子，基于权重因子和价格因子计算净值 |
| `CalcPortfolioReturn` | `PanelOperator` | 投资组合收益率计算算子 |
| `CalcTimingPortfolio` | `SectionOperator` | 将择时方向信号转换为仅做多的等资金目标权重 |
| `MakeTimingAccount` | `PanelOperator` | 为每个入池标的注入固定本金，独立交易、复投和出金 |
| `TimingAccountStats` | `BTNode` | 统计独立择时账户的收益、出入金、往返交易胜率和赔率 |
| `TimingAccountReport` | `ReportNode` | 生成含资金流水和择时交易统计的账户报告 |

`genAccountOutput` 函数用于将账户时间序列数据处理为包含净值、收益率和各项绩效统计的标准输出字典。

## 策略回测的三种模式

下面用同一份 HDF5 数据、回测区间和基准展示三种策略回测方式：

### 方式一：信号因子 + MakeAccount（快速模式）

适用于策略信号可以通过因子算子组合表达的场景。用户将策略逻辑表达为**信号因子**（目标权重或买卖数量），直接传给 `MakeAccount` 算子。不需要编写 Python 代码逻辑，完全通过因子算子的组合来实现策略。

**优点**：简洁、完全基于算子声明式构建、无需编程
**限制**：策略逻辑必须能用现有因子算子表达

### 方式二：继承 MakeStrategy（自定义模式）

适用于策略逻辑复杂、无法用现有算子组合表达的场景。用户继承 `MakeStrategy` 并重写 `genSignal` 方法，在每个再平衡时点自由编写 Python 代码来生成交易信号。

**优点**：完全灵活，可实现任意复杂的策略逻辑
**限制**：需要编写 Python 代码

### 方式三：方向信号 + 择时账户

适用于信号为 `1`（看多）、`0`（中性）、`-1`（看空）的择时策略。仅做多时，`0` 和 `-1` 均对应空仓。可通过 `CalcTimingPortfolio` 转为目标权重后接入普通账户，也可将方向信号直接传给 `MakeTimingAccount`，为每个标的独立分配本金，并用 `TimingAccountStats`、`TimingAccountReport` 统计交易和外部资金流。


本篇示例在临时目录中生成确定性的行情与因子数据，无需预先准备数据库。真实使用时将数据来源替换为自己的因子表，并确认日期、证券截面与信息可用时点一致。

## genAccountOutput — 账户输出数据处理

`genAccountOutput` 是一个工具函数，将账户的现金、负债、市值等时间序列处理为标准输出字典（包含净值、收益率、绩效统计等）。

```python
from QuantStudio.BackTest.Strategy.Strategy import genAccountOutput

def genAccountOutput(
    init_cash: float,                          # 初始资金
    cash_series: pd.Series,                    # 现金时间序列
    debt_series: pd.Series,                    # 负债时间序列
    account_value_series: pd.Series,           # 账户价值时间序列
    cash_record: pd.DataFrame,                 # 现金流记录
    debt_record: pd.DataFrame,                 # 负债记录
    risk_free_rate: float = 0.0                # 无风险利率
) -> dict
```

**内部处理逻辑**：
1. 从账户价值一阶差分计算收益，首期收益 = 首期账户价值 - 初始资金
2. 扣除现金流注入对收益的影响（现金流注入不视为策略收益）
3. 收益率 = 收益 / 前期账户价值的绝对值
4. 净值 = (1 + 收益率) 的累计乘积
5. 如有负债，额外计算无杠杆收益率（扣除杠杆对收益的影响）
6. 调用 `summaryStrategy` 生成年化收益率、波动率、Sharpe、最大回撤等绩效统计

`AccountStats` 在内部调用此函数处理 `MakeAccount`/`MakeStrategy` 产生的账户因子数据。

In [ ]:
# 在临时目录中创建独立的演示库，所有数据由固定随机种子生成。
demo_directory = TemporaryDirectory()
demo_root = Path(demo_directory.name)
(demo_root / "Cache").mkdir()
factor_dir = demo_root / "HDF5"
factor_dir.mkdir()
FDB = HDF5DB(args={"MainDir": factor_dir}).connect()
rng = np.random.default_rng(0)
demo_dates = list(pd.date_range("2025-01-01", "2025-04-30").to_pydatetime())
demo_ids = [f"{i:06d}.SZ" for i in range(1, 16)]
demo_shape = (len(demo_dates), len(demo_ids))
demo_price = pd.DataFrame(
    10 * np.exp(np.cumsum(rng.normal(.001, .01, demo_shape), axis=0)),
    index=demo_dates, columns=demo_ids,
)
FDB.writeData(Panel({"close": demo_price, "open": demo_price * .999,
                     "high": demo_price * 1.01, "low": demo_price * .99}),
              table_name="stock_cn_day_bar")
FDB.writeData(Panel({"if_listed": pd.DataFrame(1., index=demo_dates, columns=demo_ids)}),
              table_name="stock_cn_status")
FDB.writeData(Panel({"industry": pd.DataFrame(
    np.tile(["TMT", "Ind", "Fin"] * 5, (len(demo_dates), 1)),
    index=demo_dates, columns=demo_ids)}),
    table_name="stock_cn_industry", data_type={"industry": "string"})
FDB.writeData(Panel({name: pd.DataFrame(rng.uniform(.01, .2, demo_shape),
                                      index=demo_dates, columns=demo_ids)
                    for name in ("bp_lr", "ep_ttm")}), table_name="stock_cn_factor_value")
FDB.writeData(Panel({"close": demo_price.mean(axis=1).to_frame("000905.SH")}),
              table_name="index_cn_day_bar")

StartDT, EndDT = dt.datetime(2025, 1, 1), dt.datetime(2025, 3, 31)# 数据起止时间
TestStartDT, TestEndDT = dt.datetime(2025, 2, 1), EndDT# 测试起止时间

FT = FDB.getTable("stock_cn_day_bar")
DTRuler = FT.getDateTime(start_dt=StartDT, end_dt=EndDT)
TestDTs = FT.getDateTime(start_dt=TestStartDT, end_dt=TestEndDT)
SectionIDs = IDs = FT.getID()

# 再平衡时点序列
from QuantStudio.Tools.DateTimeFun import getMonthLastDateTime
BalanceDTs = getMonthLastDateTime(DTRuler)# 月末

## 方式一：信号因子 + MakeAccount（快速模式）

下面展示使用信号因子 + `MakeAccount` 构建等权配置 EP TTM 前 20% 股票的策略。策略信号完全通过因子算子组合来表达：`SectionRank` 排序 + `>= 0.8` 阈值筛选 + 归一化权重。

In [ ]:
# 策略测试
from QuantStudio.Core.CalcEngine import Engine
from QuantStudio.Core.Node import DTLocalContext, DTInitData
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Factor.BasicOperator import rename
import QuantStudio.Factor.FactorOperator as fo
from QuantStudio.Factor.FactorCache import FeatherFactorCache
from QuantStudio.BackTest.BackTestModel import BTReport
from QuantStudio.BackTest.Strategy.Strategy import MakeAccount, AccountStats, AccountReport

FT = FDB.getTable("stock_cn_day_bar")
Price = FT.getFactor("close")
OpenPrice = FT.getFactor("open")

FT = FDB.getTable("stock_cn_status")
IfListed = FT.getFactor("if_listed")

FT = FDB.getTable("stock_cn_factor_value")
EP = FT.getFactor("ep_ttm")

Mask = (IfListed == 1)

# 策略信号: EP TTM 属于前 20% 的股票等权配置
StrategySignal = fo.SectionRank(ascending=True, uniformization=True)(EP, mask=Mask, factor_args={"CalcDTRuler": BalanceDTs}) >= 0.8
StrategySignal = rename(StrategySignal / fo.Aggregate(aggr_func=np.nansum)(StrategySignal), factor_name="Signal")# 归一化

# 账户因子
Account = MakeAccount(signal_type="目标权重", init_cash=1e6, start_dt=TestDTs[0])(last_price=Price, signal=StrategySignal)

# 基准因子
FT = FDB.getTable("index_cn_day_bar")
BmkNV = FT.getFactor("close", args={"SectionIDs": ["000905.SH"]})

# 策略报告
AccountStatsNode = AccountStats(account=Account, bmk_nv=BmkNV)
AccountReportNode = AccountReport(AccountStatsNode)

Report = BTReport(report_nodes=[AccountReportNode])

with FeatherFactorCache(args={"DTRuler": DTRuler, "CacheDir": demo_root / "Cache", "StartMode": "new"}) as Cache:
    with FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Rslt = ExecEngine.run([Report], Context, fwd_data_list=[DTLocalContext(DTs=TestDTs)], init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))])

In [ ]:
from IPython.display import HTML, display

display(HTML(Rslt[0]["Report"]))

## 方式二：继承 MakeStrategy（自定义模式）

下面展示通过继承 `MakeStrategy` 并重写 `genSignal` 方法实现自定义策略——每期买入 EP TTM 最大的那只股票。这种方式适用于策略逻辑无法用现有因子算子表达的场景。

In [ ]:
# 策略测试
from QuantStudio.Core.CalcEngine import Engine
from QuantStudio.Core.Node import DTLocalContext, DTInitData
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Factor.BasicOperator import rename
import QuantStudio.Factor.FactorOperator as fo
from QuantStudio.Factor.FactorCache import FeatherFactorCache
from QuantStudio.BackTest.BackTestModel import BTReport
from QuantStudio.BackTest.Strategy.Strategy import MakeStrategy, AccountStats, AccountReport

class MakeMyStrategy(MakeStrategy):
    def genSignal(self, f, idt, x, last_price, cash, position_num, args):
        """买入 EP 最大的那只股票"""
        iID = x[0].iloc[0].idxmax()
        return pd.Series(1, index=[iID])

FT = FDB.getTable("stock_cn_day_bar")
Price = FT.getFactor("close")

FT = FDB.getTable("stock_cn_status")
IfListed = FT.getFactor("if_listed")

FT = FDB.getTable("stock_cn_factor_value")
EP = FT.getFactor("ep_ttm")

# 策略因子
Strategy = MakeMyStrategy(signal_type="目标权重", init_cash=1e6, start_dt=TestDTs[0], x_lookback=[0], x_section_ids=[None])(EP, last_price=Price)

# 基准因子
FT = FDB.getTable("index_cn_day_bar")
BmkNV = FT.getFactor("close", args={"SectionIDs": ["000905.SH"]})

# 策略报告
AccountStatsNode = AccountStats(account=Strategy, bmk_nv=BmkNV)
AccountReportNode = AccountReport(AccountStatsNode)

Report = BTReport(report_nodes=[AccountReportNode])

CacheDir = demo_root / "Cache"
if not CacheDir.exists(): CacheDir.mkdir(parents=True)

with FeatherFactorCache(args={"DTRuler": DTRuler, "CacheDir": CacheDir, "StartMode": "new"}) as Cache:
    with FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Rslt = ExecEngine.run([Report], Context, fwd_data_list=[DTLocalContext(DTs=TestDTs)], init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))])

In [ ]:
from IPython.display import HTML, display

display(HTML(Rslt[0]["Report"]))

## 方式三：方向信号+择时账户

收盘价的 5 日均线不低于 20 日均线时看多，否则看空；将方向信号滞后一个时点，使用前一时点收盘信息在本期开盘成交，本期收盘价用于估值。以下同时构造两种资金口径的账户，单边交易费率均为 `0.03%`：

- **目标权重账户**：初始总资金 100 万元。每期将资金等分给全部 `Mask=True` 的标的，仅看多的标的持仓，其他份额保留现金；池内标的数量或账户权益变化会带来再平衡。
- **独立资金账户**：每个标的入池时注入 10 万元，资金互不借用；看多时投入本子账户可用现金，连续看多不主动再平衡，平仓后的盈亏留在子账户复投。出池清仓后提走剩余权益，再入池重新注入本金。

两种账户共用信号、成交价、标的池和基准，资金规模和资金管理方式不同。数据区间内的实际进出池以 `Mask` 为准，不人为安排出入金。

In [ ]:
from QuantStudio.BackTest.Strategy.TimingStrategy import (
    CalcTimingPortfolio, MakeTimingAccount, TimingAccountStats, TimingAccountReport,
)

# 用历史收盘价构造择时信号。
MA5 = fo.RollingMean(window=5, min_periods=5)(Price)
MA20 = fo.RollingMean(window=20, min_periods=20)(Price)
TimingSignal = rename(
    fo.Lag(lag_period=1)(fo.Where()(1, MA5 >= MA20, -1)),
    factor_name="5/20日均线择时信号",
)

# 目标权重路径：全部入池标的均分总资金，仅看多份额持仓。
TimingPortfolio = CalcTimingPortfolio(descriptor_ids=SectionIDs)(
    signal=TimingSignal, mask=Mask,
)
WeightedTimingAccount = MakeAccount(
    signal_type="目标权重", init_cash=1e6, short_allowed=False, start_dt=TestDTs[0],
)(
    last_price=Price, signal=TimingPortfolio,
    buy_price=OpenPrice, sell_price=OpenPrice, buy_fee=.0003, sell_fee=.0003,
)
WeightedTimingStatsNode = AccountStats(account=WeightedTimingAccount, bmk_nv=BmkNV)
WeightedTimingReportNode = AccountReport(
    WeightedTimingStatsNode, args={"Name": "等资金择时账户报告"},
)

# 独立资金路径：方向信号直接传入，每个标的独立入金、复投和出金。
IndependentTimingAccount = MakeTimingAccount(init_cash=100_000, start_dt=TestDTs[0])(
    last_price=Price, signal=TimingSignal, mask=Mask,
    buy_price=OpenPrice, sell_price=OpenPrice, buy_fee=.0003, sell_fee=.0003,
    factor_args={"Name": "5/20日均线独立择时账户"},
)
TimingStatsNode = TimingAccountStats(
    account=IndependentTimingAccount, bmk_nv=BmkNV, args={"AccountSection": SectionIDs},
)
TimingReportNode = TimingAccountReport(
    TimingStatsNode, args={"Name": "独立择时账户报告", "ShowDetails": False},
)
TimingReport = BTReport(report_nodes=[WeightedTimingReportNode, TimingReportNode])

In [ ]:
# 沿用前文的缓存、时间标尺、标的截面和回测区间，在同一次计算中生成两份报告。
with FeatherFactorCache(args={"DTRuler": DTRuler, "CacheDir": CacheDir, "StartMode": "new"}) as Cache:
    with FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            TimingRslt, = ExecEngine.run(
                [TimingReport], Context,
                fwd_data_list=[DTLocalContext(DTs=TestDTs)],
                init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))],
            )

In [ ]:
from IPython.display import HTML, display

display(HTML(TimingRslt["Report"]))

## CalcPortfolioNV — 投资组合净值计算算子

`CalcPortfolioNV` 是一个 `PanelOperator`，基于投资组合权重因子和价格因子，通过向量化回测计算各组净值序列。常用于分位数组合回测或多策略对比场景。

### 构造参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `descriptor_ids` | `List[str]` | — | 依赖因子的截面 ID 序列 |
| `start_dt` | `Optional[datetime]` | `None` | 净值开始日，None 表示从计算的第一个时点开始 |
| `calc_type` | `Literal["numpy", "pandas"]` | `"numpy"` | 计算方式，numpy 向量化更快 |

### `__call__` 参数

| 参数 | 类型 | 说明 |
|------|------|------|
| `*portfolio` | `Factor` | 待计算净值的投资组合权重因子（至少 1 个），每个时点的因子值为各证券的资金权重 |
| `price` | `Factor` | 证券价格/净值因子 |
| `init_nv` | `float \| Factor` | 初始净值，默认 1 |
| `fee_rate` | `float \| Factor` | 交易费率，默认 0 |
| `portfolio_name_list` | `Optional[List[str]]` | 组合名称列表，None 时自动生成（取因子名或 `P0, P1, ...`） |
| `factor_args` | `dict` | 传递给净值因子的参数 |

**返回值**：净值因子，列为各组合的净值序列。

> **注意**：`portfolio_name_list` 会自动按升序排列组合；若有重复名称，系统会使用 `P0, P1, ...` 替代。

### 构造示例

```python
from QuantStudio.BackTest.Strategy.AllocationStrategy import CalcPortfolioNV

calcPortfolioNV = CalcPortfolioNV(descriptor_ids=SectionIDs, start_dt=TestDTs[0])
PortfolioNV = calcPortfolioNV(
    *QuantilePortfolios,   # 多个权重因子
    price=Price,
    init_nv=1
)
```

## 输出解读

`AccountReport`（报告节点）生成的 HTML 报告按以下顺序展示：

### 报告内容

1. **参数设置** — 显示回测参数：无风险利率、再平衡时点等
2. **账户表现图** — 账户价值曲线（折线）+ 收益柱状图（双轴）
3. **净值表现图** — 净值曲线 + 收益率柱状图 + 基准净值对比（如有）
4. **考虑资金投入的表现**（如有现金流注入）
5. **相对表现图**（如有基准）— 相对净值曲线 + 相对收益率柱状图
6. **绩效统计表** — 包含绝对表现和相对表现的所有指标

### 输出字典结构

`AccountStats` 的 `backward_compute` 返回字典可通过编程方式访问各维度数据（`AccountReport` 会自动调用 `AccountStats` 的输出生成报告）：

```python
# 访问账户报告输出
account_output = Output["0-账户报告"]

# 时间序列数据
ts = account_output["时间序列"]
nv = ts["净值"]               # 策略净值序列
returns = ts["收益率"]         # 日收益率序列
account_value = ts["账户价值"] # 账户总价值序列

# 绩效统计数据
stats = account_output["统计数据"]
annual_return = stats.loc["年化收益率"]    # 年化收益率
sharpe = stats.loc["Sharpe比率"]           # Sharpe 比率
max_dd = stats.loc["最大回撤"]             # 最大回撤
win_rate = stats.loc["胜率"]               # 胜率

# 相对表现（如有基准）
if "相对表现" in stats.columns:
    excess_return = stats["相对表现"].loc["年化收益率"]
    ir = stats["相对表现"].loc["信息比率"]
```

## 策略回测完整流程总结

```
                    因子算子组合                    重写 genSignal
                         │                              │
                         ▼                              ▼
                  信号因子 (Factor)              MakeMyStrategy
                         │                              │
                         └──────────┬───────────────────┘
                                    │
                                    ▼
                            MakeAccount.__call__
                           （撮合成交、更新持仓）
                                    │
                                    ▼
                            账户因子 (Factor)
                           [Cash, Position, Amount, Signal, TradeNum, TradePrice, Fee]
                                    │
                                    ▼
                          AccountStats
                           （净值计算、绩效统计）
                                    │
                                    ▼
                            AccountReport
                           （生成 HTML 报告）
                                    │
                                    ▼
                              HTML 报告 + 统计数据
```

构造和执行两个阶段的详细说明请参阅 **[基本框架](基本框架.ipynb)**。

择时策略的目标权重路径也遵循上述流程；独立资金路径使用 `MakeTimingAccount → TimingAccountStats → TimingAccountReport`。两条路径在“方式三”中共享行情、方向信号和计算上下文，并由 `BTReport` 汇总。

## AccountStats — 账户统计节点

`AccountStats` 是 `BTNode` 子类，接收账户因子（由 `MakeAccount` 或 `MakeStrategy` 产生），计算净值、收益率和绩效统计指标。

### 参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"账户统计"` | 节点名称（冻结） |
| `RiskFreeRate` | `float` | `0` | 无风险利率，用于计算 Sharpe 比率（冻结） |
| `RebalanceDTs` | `Optional[List[datetime]]` | `None` | 再平衡时点，用于计算换手率（冻结） |

### 构造方法

```python
AccountStats(
    account: Factor,                # 账户因子（由 MakeAccount 或 MakeStrategy 产生）
    bmk_nv: Optional[Factor]=None,  # 基准净值因子，用于计算相对表现指标
    args={}, config_file=None, **kwargs
)
```

### 输出数据结构

`backward_compute` 返回的字典包含以下键：

| 键 | 类型 | 说明 |
|------|------|------|
| `"时间序列"` | `DataFrame` | 包含现金、负债、证券市值、账户价值、收益、收益率、累计收益率、净值等列 |
| `"统计数据"` | `DataFrame` | 汇总绩效指标 |
| `"现金流记录"` | `DataFrame` | 每笔现金流明细 |
| `"负债记录"` | `DataFrame` | 每笔融资明细 |

**时间序列列说明**：

| 列 | 说明 |
|------|------|
| `现金` | 账户剩余现金 |
| `证券` | 持仓证券市值 |
| `账户价值` | 现金 + 证券 - 负债 |
| `收益` | 账户价值变动（扣除现金流注入） |
| `收益率` | 收益 / 前期账户价值的绝对值 |
| `净值` | (1 + 收益率) 的累计乘积 |
| `考虑资金投入的累计收益率` | （如有现金流注入）账户价值 / 累计资金投入 - 1 |
| `无杠杆收益率` | （如有负债）扣除杠杆后的收益率 |

**统计数据指标**：

| 指标 | 说明 |
|------|------|
| 年化收益率 | 累计收益率的年化值 |
| 年化波动率 | 收益率标准差的年化值 |
| Sharpe 比率 | 年化超额收益 / 年化波动率 |
| 最大回撤 | 净值曲线上的最大回撤幅度 |
| Calmar 比率 | 年化收益率 / 最大回撤 |
| 胜率 | 收益率为正的交易日占比 |
| 盈亏比 | 正收益均值 / 负收益均值（取绝对值） |

若提供了基准净值 `bmk_nv`，额外包含**相对表现**统计：
- 年化超额收益率
- 跟踪误差
- 信息比率
- 超额收益胜率

## AccountReport — 账户报告节点

`AccountReport` 是 `ReportNode` 子类，接收 `AccountStats` 节点的输出，生成 HTML 报告。

### 构造方法

```python
AccountReport(
    account_stats: AccountStats,  # AccountStats 节点实例
    args={}, config_file=None, **kwargs
)
```

### 报告可视化

`AccountReport` 生成的 HTML 包含以下图表：
- **账户价值**曲线 + 收益柱状图
- **净值曲线**（含基准净值对比）
- **考虑资金投入的表现**（如有现金流）
- **相对净值**和**相对收益率**（如有基准）

## MakeStrategy — 策略创建算子（可继承）

`MakeStrategy` 继承自 `MakeAccount`，允许用户通过重写 `genSignal` 方法实现自定义策略逻辑。当策略信号无法用现有因子算子表达时，继承此类是最灵活的方式。

### 构造参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `signal_type` | `Literal["买卖数量", "目标权重"]` | `"目标权重"` | 信号类型，含义同 `MakeAccount` |
| `init_cash` | `float` | `1e6` | 初始资金 |
| `short_allowed` | `bool` | `False` | 是否允许卖空 |
| `start_dt` | `Optional[datetime]` | `None` | 净值开始日，None 表示从计算的第一个时点开始 |
| `x_lookback` | `List[int]` | `[]` | 策略所依赖因子的回溯期数列表 |
| `x_section_ids` | `List[Optional[List[str]]]` | `[]` | 策略所依赖因子的截面 ID 序列列表 |
| `signal_dts` | `Optional[List[datetime]]` | `None` | 信号生成时点，None 表示每个时点都生成信号 |

> **注意**：`x_lookback` 和 `x_section_ids` 长度必须相等，且与 `__call__` 中传入的 `*x` 因子个数相同。

### `genSignal` — 策略信号生成方法（必须重写）

```python
def genSignal(
    self,
    f: PanelOperation,                  # 策略因子对象
    idt: dt.datetime,                   # 当前时点
    x: List[pd.DataFrame],              # 策略依赖的因子数据，每个元素为 DataFrame(index=[datetime], columns=[证券ID])
    last_price: pd.Series,              # 当前证券最新价，Series(index=[证券ID])
    cash: float,                        # 当前账户剩余现金
    position_num: pd.Series,            # 当前账户持仓数量，Series(index=[证券ID])
    args: dict                          # 额外参数字典
) -> None | pd.Series:
```

**返回值**：
- `None` — 当前时点不产生交易信号
- `pd.Series` — 交易信号，index 为证券 ID。含义取决于 `signal_type`：
  - `"买卖数量"`：正值为买入股数，负值为卖出股数
  - `"目标权重"`：目标持仓权重（算子内部会转换为买卖数量）

### `__call__` 参数

| 参数 | 类型 | 说明 |
|------|------|------|
| `*x` | `Factor` | 策略依赖的因子列表，按顺序对应 `x_lookback` 和 `x_section_ids` |
| `last_price` | `Factor` | 最新价因子 |
| `init_account` | `Optional[Factor]` | 初始账户因子，None 时自动创建 |
| `buy_price` / `buy_limit` / `buy_fee` / `buy_amt_limit` | — | 交易限制参数，同 `MakeAccount.__call__` |
| `sell_price` / `sell_limit` / `sell_fee` / `sell_amt_limit` | — | 交易限制参数，同 `MakeAccount.__call__` |
| `extra_deps` | `List[Node]` | 额外依赖的计算节点 |
| `extra_section_ids` | `List[Optional[List[str]]]` | 额外依赖节点的截面 ID |
| `extra_lookback` | `List[int]` | 额外依赖节点的回溯期数 |
| `factor_args` | `dict` | 传递给策略因子的参数 |

### 构造示例

```python
from QuantStudio.BackTest.Strategy.Strategy import MakeStrategy

class MakeMyStrategy(MakeStrategy):
    def genSignal(self, f, idt, x, last_price, cash, position_num, args):
        """买入 EP 最大的那只股票"""
        iID = x[0].iloc[0].idxmax()
        return pd.Series(1, index=[iID])

# 创建策略因子
Strategy = MakeMyStrategy(
    signal_type="目标权重",
    init_cash=1e6,
    start_dt=TestDTs[0],
    x_lookback=[0],
    x_section_ids=[None]
)(EP, last_price=Price)
```

## MakeAccount — 账户创建算子

`MakeAccount` 是一个 `PanelOperator`（面板算子），在每个时点根据交易信号撮合成交，更新账户的现金和持仓。

### 构造参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `signal_type` | `Literal["买卖数量", "目标权重"]` | `"目标权重"` | 信号类型：买卖数量（直接指定股数）或目标权重（指定目标仓位比例） |
| `init_cash` | `float` | `1e6` | 初始资金 |
| `short_allowed` | `bool` | `False` | 是否允许卖空 |
| `start_dt` | `Optional[datetime]` | `None` | 净值开始日，None 表示从计算的第一个时点开始 |

### `__call__` 参数

| 参数 | 类型 | 说明 |
|------|------|------|
| `last_price` | `Factor` | 最新价因子，用于计算持仓市值和成交金额 |
| `signal` | `Factor` | 信号因子，含义取决于 `signal_type`：目标权重（可小于满仓，未投资部分保留现金）或买卖数量（正数为买入股数，负数为卖出股数） |
| `target_price` | `Optional[Factor]` | 目标价因子，None 表示信号转换为市价单 |
| `init_account` | `Optional[Factor]` | 初始账户因子（复合因子），None 时使用 `init_cash` 自动创建 |
| `buy_price` | `Optional[Factor]` | 买入成交价因子，None 时使用 `last_price` |
| `buy_limit` | `Optional[Factor]` | 禁止买入条件因子，值为 1 的 ID 禁止买入 |
| `buy_fee` | `float \| Factor` | 买入交易费率，默认 0 |
| `buy_amt_limit` | `Optional[Factor]` | 买入成交额限制因子，该期买入额不能超过该因子值 |
| `sell_price` | `Optional[Factor]` | 卖出成交价因子，None 时使用 `last_price` |
| `sell_limit` | `Optional[Factor]` | 禁止卖出条件因子，值为 1 的 ID 禁止卖出 |
| `sell_fee` | `float \| Factor` | 卖出交易费率，默认 0 |
| `sell_amt_limit` | `Optional[Factor]` | 卖出成交额限制因子，该期卖出额不能超过该因子值 |
| `factor_args` | `dict` | 传递给账户因子的参数（如 `CalcDTRuler` 指定计算时点） |

**返回值**：产生一个复合类型因子，每行包含 7 个子字段：

| 子字段 | 类型 | 说明 |
|--------|------|------|
| `Cash` | `double` | 剩余现金 |
| `Position` | `double` | 持仓数量 |
| `Amount` | `double` | 持仓金额 |
| `Signal` | `double` | 交易信号（原始输入） |
| `TradeNum` | `double` | 交易数量，正值表示买入，负值表示卖出 |
| `TradePrice` | `double` | 平均成交价 |
| `Fee` | `double` | 交易费用 |

### 撮合逻辑

账户算子在每个时点执行以下撮合流程：

1. **信号转换**：若 `signal_type="目标权重"`，将目标权重转换为买卖数量（基于当前账户市值和最新价）
2. **交易限制过滤**：检查买入/卖出禁止条件、成交额限制、卖空限制
3. **先卖后买**：先执行卖出交易释放现金，再执行买入交易（受可用现金约束）
4. **未成交撤单**：超出交易限制的市价单自动撤销
5. **小额清零**：持仓数量 < `1e-8` 或现金 < `1e-8` 的尾差自动清零

### 构造示例

```python
from QuantStudio.BackTest.Strategy.Strategy import MakeAccount

# 信号因子 + MakeAccount 创建账户
Account = MakeAccount(
    signal_type="目标权重",
    init_cash=1e6,
    start_dt=TestDTs[0]
)(
    last_price=Price,
    signal=StrategySignal
)
```

## CalcTimingPortfolio — 仅做多择时策略

位于 `QuantStudio.BackTest.Strategy.TimingStrategy`，也可通过 `QuantStudio.BackTest.api.Strategy.TimingStrategy` 访问。

方向信号为 `1`（看多）、`0`（中性）、`-1`（看空）。每个交易时点先将账户总资金等分给全部 `mask=True` 的标的，只有信号为 `1` 的标的买入，其他标的分配的资金保留为现金。例如池中有 4 个标的、其中 1 个看多，该标的目标权重为 25%，现金为 75%。

- `descriptor_ids`：完整标的截面，默认 `None` 使用输出截面。只输出部分标的时，显式提供完整截面以保持等分资金的分母。
- `signal`：择时方向因子，缺失信号按空仓处理。
- `mask`：标的池因子，True 或 1 表示入选；默认 `None` 表示全部标的入选，缺失值表示未入选。
- `factor_args`：组合因子参数，`CalcDTRuler` 可指定交易时点，默认每个计算时点都生成目标权重。

`0`、`-1`、缺失信号或移出池的标的均输出零权重；空池和全池无看多信号时输出全零，以触发清仓。非交易时点输出 NaN，表示不调仓。成交仍受 `MakeAccount` 的价格、费用和买卖限制约束。

输出是目标权重，可接入 `MakeAccount` 或 `CalcPortfolioNV`。方向信号使用当期输入，在当期价格成交；信号若在收盘后才可得，应由调用方先滞后信号，或使用后续可成交价格。前面“方式三”中的 `TimingPortfolio` 复用了前文行情产生的 `TimingDirection` 和上市标的池 `Mask`，并通过 `WeightedTimingAccount` 完成回测。完整截面、时间标尺和回测区间均与前两种方式一致。

## TimingStrategy.MakeTimingAccount — 固定本金的独立择时账户

`TimingStrategy.MakeTimingAccount` 接收原始方向信号：`1` 看多，`0`、`-1` 或缺失值空仓。每个标的首次 `mask=True` 时独立注入 `init_cash`（默认 100 万），资金互不借用。看多时投入该标的可用现金，连续看多不调整已有持仓；平仓后的盈亏留在原子账户，下次看多继续复投。其他标的入池不会改变已有标的的仓位。

标的变为 `mask=False` 或缺失时清仓并提走该子账户的全部剩余现金，重新入池时重新注入固定本金。卖出受限时继续保留账户，待实际清仓后出金；在清仓前重新入池则沿用原子账户。买卖报价、禁止交易条件、费率和单时点成交金额上限的含义沿用普通账户，费用由各子账户承担。缺失报价不成交，估值沿用最近有效价格。

调用方式为 `TimingStrategy.MakeTimingAccount(init_cash=..., start_dt=...)(last_price=Price, signal=Direction, mask=Mask, ...)`；`mask=None` 表示全部标的入池。`start_dt` 用于确定完整账户历史，初始时所有标的均无资金和持仓。

### 账户字段与配套统计

前七个复合字段仍为 `Cash`、`Position`、`Amount`、`Signal`、`TradeNum`、`TradePrice`、`Fee`，其中 `Cash` 是每个标的自己的现金，`Signal` 是原始方向信号。增加 `Active`（子账户资金尚未撤出）、`Deposit`（当期入金）、`Withdrawal`（当期出金）、`LastPrice`（最近有效估值价）。Deposit 和 Withdrawal 均记录正金额，净现金流为二者之差。

必须配套使用本模块的 `TimingStrategy.TimingAccountStats`，它对各子账户的现金求和，并提供 `TimingStrategy.TimingAccountReport` 展示统计与资金流水。普通账户的 `Strategy.AccountStats`、`calcAccountNV`、`calcAccountPortfolio` 按共享现金解释账户，不适用于此账户结构。

输出包含：

- `时间序列`：账户权益、入金、出金、净现金流、累计入金、累计出金、累计净投入、收益与资金流调整后的净值。同期相抵的出入金仍分别记录。
- `资金流水`（同 `现金记录`）：按时间、标的逐笔记录入池入金和出池出金。
- `子账户现金`、`子账户权益`、`持仓数量`、`交易记录`：可核对各标的独立交易及费用。
- `子账户净值`、`标的价格`：各子账户扣除外部资金流后的独立净值，以及最近有效估值价，用于分标的净值与实际交易图。
- `择时交易`、`未平仓交易`：按实际成交重建完整开仓至清仓的往返交易，分批成交合并为一笔。未平仓交易的浮动收益单独展示，不计入胜率和赔率。统计子区间时会读取账户起始以来的历史，保留原始开仓成本。
- `择时统计`：包含总计和各标的的交易次数、盈利次数、亏损次数、持平次数、胜率、赔率、平均盈亏金额、盈利因子、总盈亏、平均收益率与平均持仓天数。

胜率为盈利笔数 / 已平仓笔数，持平交易计入分母；赔率（同盈亏比）为平均盈利金额 / 平均亏损金额的绝对值，均扣除双边费用。单笔收益率以实际买入金额加买入费用为分母。没有已平仓交易时胜率为 NaN，缺少盈利或亏损样本时赔率为 NaN。

资金流按当期交易前入金、交易后出金处理：`收益 = 当期权益 - 上期权益 - 入金 + 出金`，`收益率 = 收益 / (上期权益 + 当期入金)`，无资金且无盈亏的期间收益率为 0，净值由收益率连乘。累计净投入可能在全部出池后为负，不能直接用它作净值分母。

前面“方式三”中的 `IndependentTimingAccount` 直接接收同一个 `TimingDirection` 和 `Mask`，与目标权重账户在同一个 `BTReport` 中执行。资金流水和往返交易指标通过 `TimingOutput` 读取，报告与明细导出到 `docs/data/timing_strategy/`。

## TimingStrategy.TimingAccountReport — 择时账户报告

接收 `TimingAccountStats` 节点，生成账户绩效、择时统计以及账户净值与资金流水图像。

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `ShowDetails` | `bool` | `False` | 是否展示明细。默认择时统计只保留“总计”；设为 `True` 时增加各标的统计列、独立净值及交易标记图 |
| `ReportKey` | `str` | `"Report"` | HTML 报告在结果字典中的键名 |

通过 `TimingAccountReport(TimingStatsNode, args={"ShowDetails": True})` 开启分标的统计和图像。两种设置均不展示择时交易表和未平仓交易表；该参数只控制 HTML 展示，完整的择时统计、交易与资金明细仍保留在输出字典中。

账户净值图使用已经扣除外部资金流影响的 `时间序列["净值"]`，存在 `基准净值` 时一并绘制基准。资金流水图只展示每个时点全部子账户合计的净现金流，即总入金减总出金；正值为净入金，负值为净出金，同期等额出入金相抵为零。图像以 PNG 内嵌在 HTML 中。

每个标的的明细图上方展示 `子账户净值`，下方展示 `标的价格`，并按 `交易记录` 中的实际成交时点、成交价格标注买入（上三角）和卖出（下三角）。受限而未成交的信号不产生标记，分批成交逐次标注。

子账户净值按各自的权益变动扣除入金、加回出金后计算收益率，并连乘得到；买卖费用已计入权益。出池后的无资金期间净值保持不变，再入池延续既有净值。统计从中途开始时，以该子账户上期实际权益为起点，净值从 1 开始累计。